# Avance 1. Análisis Exploratorio de Datos (EDA)

**Sistema Inteligente para la Optimización de Operaciones de Transporte mediante Inteligencia Artificial aplicada al Análisis Predictivo de Datos GPS**

Maestría en Inteligencia Artificial Aplicada — TC5035 — Grupo 56

**Equipo:**
- Alejandra Guadalupe Larrañaga Altamirano — A01794334
- Mario Alberto Guillén De La Torre — A01796701
- Andrea Kimberly Fray Tamayo — A01797168

**Fecha:** [COMPLETAR]

---

**Importante antes de entregar:** este notebook debe ejecutarse de principio a fin de manera **SECUENCIAL** (Kernel → Restart & Run All) antes de subir el link al repositorio y entregar la tarea. Si lo corres en Google Colab sin tener el repo clonado/montado, la celda de carga de datos te va a pedir subir los archivos manualmente — ver sección 0.

**Fuente de verdad del esquema de datos:** [`docs/data-schema.md`](../docs/data-schema.md) del repo. Cualquier duda sobre qué significa una columna, o qué variables NO se pueden usar como input de un modelo (regla de data leakage), está documentada ahí.

## 0. Configuración y carga de datos

Cargamos ambas fuentes por separado antes de unirlas, para poder comparar `real` vs `synthetic` explícitamente en el análisis bivariado. La función de carga funciona tanto si corres el notebook dentro del repo clonado (rutas relativas normales) como si lo abres suelto en Colab (te pedirá subir el archivo manualmente).

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

IN_COLAB = "google.colab" in sys.modules

# TODO: ajustar el nombre del archivo si Mario lo nombró distinto
SYNTHETIC_PATH = "../data/synthetic/trips_synthetic.csv"
REAL_PATH = "../data/raw/trips_real_export.csv"


def cargar_datos(path, nombre):
    """Lee el CSV desde la ruta del repo clonado. Si no existe y estamos en
    Colab (notebook abierto suelto, sin el repo), pide subirlo manualmente
    — así un Restart & Run All no truena solo por el entorno."""
    if os.path.exists(path):
        return pd.read_csv(path)
    if IN_COLAB:
        print(f"No se encontró '{path}' localmente. Sube el archivo '{nombre}':")
        from google.colab import files
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        return pd.read_csv(fname)
    raise FileNotFoundError(
        f"No se encontró {path}. Corre este notebook desde dentro del repo "
        f"clonado (carpeta notebooks/), o ábrelo en Colab para que te pida "
        f"subir el archivo manualmente."
    )


df_synthetic = cargar_datos(SYNTHETIC_PATH, "dataset sintético de trips")
df_real = cargar_datos(REAL_PATH, "export de trips reales")

assert "data_source" in df_synthetic.columns, "Falta data_source en el dataset sintético"
assert "data_source" in df_real.columns, "Falta data_source en el dataset real"

df = pd.concat([df_real, df_synthetic], ignore_index=True)

# Conversión a datetime aquí, antes de cualquier describe() — si no, pandas
# trata started_at/ended_at como texto plano en las estadísticas.
for col in ["started_at", "ended_at"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f"Reales: {len(df_real)} | Sintéticos: {len(df_synthetic)} | Total: {len(df)}")
df.head()

## 1. Estructura de los datos

*(Criterio de rúbrica — 20 pts: forma y tipos de datos, estadísticas descriptivas de todas las variables, frecuencia de clases en categóricas, valores faltantes identificados.)*

In [ ]:
print("Forma del dataset:", df.shape)
print()
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object", "category"])

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No hay valores faltantes en este punto del pipeline.")
else:
    missing_pct = (missing / len(df) * 100).round(2)
    display(pd.DataFrame({"faltantes": missing, "%": missing_pct}))

# Nota: esto se vuelve a revisar DESPUÉS de la sección 1.5, porque derivar
# previous_trip_duration y historical_corridor_mean_duration va a introducir
# NaNs estructurales (primer viaje de cada vehículo/corredor) que todavía
# no existen aquí.

In [ ]:
for col in ["data_source", "origin_zone", "destination_zone"]:
    if col in df.columns:
        print(f"--- {col} ({df[col].nunique()} categorías) ---")
        print(df[col].value_counts())
        print()

## 1.5 Derivación de variables

Estas columnas son features candidatas para el modelo (ver sección 4.4) pero no necesariamente vienen ya calculadas en el CSV crudo — se derivan aquí, una sola vez, antes de cualquier análisis que las use.

**Importante — mismo principio de leakage que ya aplicamos en el backend (`TripBuilderService`):** `previous_trip_duration` y `historical_corridor_mean_duration` deben calcularse ordenando por tiempo y usando solo información *anterior* a cada viaje (`.shift(1)` / `.expanding().shift(1)`), nunca el dataset completo de una sola vez — eso sería fuga de información hacia el pasado.

In [ ]:
# Corredor como combinación origin→destination — se usa para el histórico
# por corredor y para medir su cardinalidad en la sección 4.3.
if "origin_zone" in df.columns and "destination_zone" in df.columns:
    df["corridor"] = df["origin_zone"] + " → " + df["destination_zone"]

# hour_of_day / day_of_week — derivadas de started_at si no vienen ya en el CSV
if "hour_of_day" not in df.columns and "started_at" in df.columns:
    df["hour_of_day"] = df["started_at"].dt.hour
if "day_of_week" not in df.columns and "started_at" in df.columns:
    df["day_of_week"] = df["started_at"].dt.dayofweek  # 0=lunes

# previous_trip_duration — viaje inmediatamente anterior del MISMO vehículo,
# ordenado por tiempo. IMPORTANTE: agrupamos por (data_source, vehicle_id),
# no solo vehicle_id — si un vehicle_id sintético coincide numéricamente con
# uno real, agrupar solo por vehicle_id mezclaría el historial de ambas
# fuentes, que son unidades distintas aunque compartan ID.
if "previous_trip_duration" not in df.columns and "vehicle_id" in df.columns:
    df = df.sort_values("started_at")
    df["previous_trip_duration"] = df.groupby(["data_source", "vehicle_id"])["duration_seconds"].shift(1)

# historical_corridor_mean_duration — promedio histórico del corredor, SOLO
# con viajes anteriores (expanding + shift), y SOLO dentro de la misma fuente
# (data_source) — mismo motivo que arriba: un corredor real y uno sintético
# no deberían promediarse juntos.
if "historical_corridor_mean_duration" not in df.columns and "corridor" in df.columns:
    df = df.sort_values("started_at")
    df["historical_corridor_mean_duration"] = (
        df.groupby(["data_source", "corridor"])["duration_seconds"]
          .apply(lambda s: s.expanding().mean().shift(1))
          .reset_index(level=[0, 1], drop=True)
    )

nuevas_nan = df[["previous_trip_duration", "historical_corridor_mean_duration"]].isnull().sum()
print("NaN introducidos por derivación (esperado — primer viaje de cada vehículo/corredor DENTRO de su fuente):")
print(nuevas_nan)

## 2. Análisis univariante

*(Criterio de rúbrica — 20 pts: histogramas, boxplots, gráficos de barras, **según el tipo de dato y escala de medición** de cada variable — la rúbrica lo pide explícitamente, así que variables discretas/ordinales van con gráfico de barras, no histograma.)*

In [ ]:
print("Asimetría (skew) de duration_seconds:", df["duration_seconds"].skew().round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["duration_seconds"], kde=True, ax=axes[0, 0])
axes[0, 0].set_title("duration_seconds (escala original)")
sns.histplot(np.log1p(df["duration_seconds"]), kde=True, ax=axes[0, 1])
axes[0, 1].set_title("log(duration_seconds + 1)")
sns.boxplot(x=df["duration_seconds"], ax=axes[1, 0])
axes[1, 0].set_title("Boxplot duration_seconds")
sns.boxplot(x=np.log1p(df["duration_seconds"]), ax=axes[1, 1])
axes[1, 1].set_title("Boxplot log(duration_seconds + 1)")
plt.tight_layout()
plt.show()

# TODO: con el valor de skew() y la comparación visual, ¿conviene modelar
# en escala log? Justificar en la sección de Conclusiones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["distance_km"], kde=True, ax=axes[0])
axes[0].set_title("Distribución de distance_km")
sns.boxplot(x=df["distance_km"], ax=axes[1])
axes[1].set_title("Boxplot de distance_km")
plt.tight_layout()
plt.show()

In [ ]:
# hour_of_day y day_of_week son discretas/ordinales -> gráfico de barras,
# no histograma (así lo pide la rúbrica según la escala de medición).
# reindex(range(7)) asegura que los 7 días aparezcan aunque alguno tenga
# 0 viajes — sin esto, si falta un día, el gráfico tiene menos barras que
# etiquetas y la celda truena.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["hour_of_day"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Viajes por hora del día")
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
df["day_of_week"].value_counts().reindex(range(7), fill_value=0).plot(kind="bar", ax=axes[1])
axes[1].set_xticklabels(dias, rotation=0)
axes[1].set_title("Viajes por día de la semana")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
df["origin_zone"].value_counts().plot(kind="barh", ax=axes[0])
axes[0].set_title("Viajes por zona de origen")
df["destination_zone"].value_counts().plot(kind="barh", ax=axes[1])
axes[1].set_title("Viajes por zona de destino")
plt.tight_layout()
plt.show()

## 3. Análisis bi/multivariante

*(Criterio de rúbrica — 20 pts: correlación cuantitativa Y gráfica entre variables importantes, incluyendo la variable objetivo.)*

In [ ]:
# Excluimos IDs explícitamente — no son variables numéricas con significado
# de magnitud, son identificadores, no deben entrar a una matriz de correlación.
id_cols = [c for c in df.columns if c.endswith("_id")]
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in id_cols]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.heatmap(df[numeric_cols].corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Correlación de Pearson (relaciones lineales)")
sns.heatmap(df[numeric_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlación de Spearman (relaciones monótonas, robusta a sesgo)")
plt.tight_layout()
plt.show()

# TODO: ¿alguna correlación es sospechosamente alta con duration_seconds?
# Contrastar contra la lista de variables prohibidas en docs/data-schema.md
# antes de considerarla como feature — una correlación perfecta ahí sería
# señal de leakage, no de buena feature.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="distance_km", y="duration_seconds", hue="data_source", alpha=0.6)
plt.title("Distancia vs duración, real vs sintético")
plt.tight_layout()
plt.show()

In [ ]:
# Duración por zona de destino — parte GRÁFICA del análisis bivariado categórico
plt.figure(figsize=(12, 6))
order = df.groupby("destination_zone")["duration_seconds"].median().sort_values().index
sns.boxplot(data=df, x="destination_zone", y="duration_seconds", order=order)
plt.xticks(rotation=75)
plt.title("Duración del viaje por zona de destino")
plt.tight_layout()
plt.show()

In [ ]:
# Parte CUANTITATIVA del análisis bivariado categórico (la rúbrica pide
# ambas, no solo la gráfica): tabla de mediana/IQR por zona + prueba de
# Kruskal-Wallis (no asume normalidad, apropiado si duration_seconds está sesgada).
tabla_zonas = df.groupby("destination_zone")["duration_seconds"].agg(
    mediana="median",
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
    n="count",
).sort_values("mediana")
display(tabla_zonas)

grupos = [g["duration_seconds"].values for _, g in df.groupby("destination_zone") if len(g) > 1]
if len(grupos) > 1:
    h_stat, p_value = stats.kruskal(*grupos)
    print(f"Kruskal-Wallis: H={h_stat:.2f}, p-value={p_value:.4f}")
    # TODO: interpretar — p-value bajo sugiere que la zona de destino sí
    # afecta la duración de forma estadísticamente significativa.

In [ ]:
# Tendencia temporal — la rúbrica pregunta por esto explícitamente, y el
# generador sintético de Mario simula un factor de tráfico por hora/día,
# así que debería verse reflejado aquí.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=df, x="hour_of_day", y="duration_seconds", hue="data_source",
             estimator="median", errorbar=None, ax=axes[0])
axes[0].set_title("Duración mediana por hora del día")
sns.boxplot(data=df, x="day_of_week", y="duration_seconds", order=range(7), ax=axes[1])
axes[1].set_xticklabels(dias, rotation=0)
axes[1].set_title("Duración por día de la semana")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(data=df, x="duration_seconds", hue="data_source", fill=True, common_norm=False, ax=ax)
ax.set_title("Duración del viaje: distribución real vs sintética")
plt.tight_layout()
plt.show()

# TODO: con solo 7 viajes reales esta comparación es más ilustrativa
# que estadísticamente concluyente — decirlo explícitamente en conclusiones,
# no presentarlo como una validación robusta.

## 4. Preprocesamiento

*(Criterio de rúbrica — 20 pts: ejecutar Y justificar las estrategias para valores faltantes, atípicos, y alta cardinalidad.)*

### 4.1 Valores faltantes

Revisamos otra vez, ahora que la sección 1.5 ya introdujo los NaN estructurales esperados (primer viaje por vehículo/corredor).

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No hay valores faltantes.")
else:
    missing_pct = (missing / len(df) * 100).round(2)
    display(pd.DataFrame({"faltantes": missing, "%": missing_pct}))

# ¿Los faltantes se concentran en una fuente (real vs synthetic)?
display(df.groupby("data_source").apply(lambda g: g.isnull().mean().round(3)))

# TODO: decisión — para previous_trip_duration/historical_corridor_mean_duration
# en el primer viaje, ¿imputar con un valor global, dejar NaN y que el modelo
# lo maneje (algunos algoritmos sí toleran NaN), o descartar esas filas?
# Justificar la elección.

In [ ]:
# APLICAR la decisión (default razonable, ajustar y justificar en texto):
# imputamos con la MEDIANA del grupo (data_source, corridor) cuando existe
# suficiente historial; si no hay ninguno, con la mediana global. Es un
# default conservador — si deciden algo distinto (ej. dejar NaN para que
# el modelo lo maneje, o eliminar esas filas), reemplacen este bloque.
df_model = df.copy()

for col in ["previous_trip_duration", "historical_corridor_mean_duration"]:
    mediana_por_corredor = df_model.groupby(["data_source", "corridor"])[col].transform("median")
    df_model[col] = df_model[col].fillna(mediana_por_corredor)
    df_model[col] = df_model[col].fillna(df_model[col].median())

print("Faltantes después de imputar:")
print(df_model[["previous_trip_duration", "historical_corridor_mean_duration"]].isnull().sum())

### 4.2 Valores atípicos

En datos GPS los atípicos típicos no son solo "duración muy larga" — también hay saltos de señal: velocidades imposibles, viajes de pocos segundos, distancias absurdas para la duración registrada. Combinamos IQR (sobre la escala log, porque la duración está sesgada) con reglas de dominio.

In [ ]:
# IQR sobre log — más robusto que IQR puro cuando la variable está sesgada
log_duration = np.log1p(df["duration_seconds"])
Q1, Q3 = log_duration.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers_iqr = df[(log_duration < lower) | (log_duration > upper)]

# Reglas de dominio — TODO: ajustar los umbrales según lo que tenga sentido
# para traslados turísticos reales (ej. ¿qué velocidad máxima es físicamente
# plausible en el corredor? ¿qué duración mínima es un viaje real vs ruido?)
UMBRAL_VELOCIDAD_MAX = 120  # km/h
UMBRAL_DURACION_MIN = 60    # segundos

outliers_dominio = df[
    (df.get("max_speed", 0) > UMBRAL_VELOCIDAD_MAX) |
    (df["duration_seconds"] < UMBRAL_DURACION_MIN)
]

print(f"Atípicos por IQR (log): {len(outliers_iqr)} de {len(df)}")
print(f"Atípicos por regla de dominio: {len(outliers_dominio)} de {len(df)}")
print()
print("Atípicos por fuente (importante: con solo 7 viajes reales, eliminar")
print("uno es perder ~14% de esa muestra — pensarlo dos veces antes de descartar):")
print(pd.concat([outliers_iqr, outliers_dominio]).drop_duplicates()["data_source"].value_counts())

# TODO: decidir qué hacer con cada grupo y justificar — no es lo mismo un
# atípico sintético (se puede simplemente regenerar) que uno real (dato escaso).

In [ ]:
# APLICAR la decisión (default razonable, ajustar y justificar en texto):
# en vez de ELIMINAR filas (perder datos reales escasos es caro), hacemos
# CAPPING (winsorizing) de duration_seconds a los límites del IQR-log —
# conserva la fila pero limita el valor extremo. Si deciden eliminar en vez
# de acotar (ej. para los que violan reglas de dominio, más probablemente
# ruido de GPS que viajes reales), ajusten aquí.
lower_original = np.expm1(lower)
upper_original = np.expm1(upper)

df_model["duration_seconds"] = df_model["duration_seconds"].clip(lower_original, upper_original)
df_model = df_model.drop(index=outliers_dominio.index, errors="ignore")

print(f"Filas después de preprocesar atípicos: {len(df_model)} (de {len(df)} originales)")

### 4.3 Alta cardinalidad

In [ ]:
print("Categorías en origin_zone:", df["origin_zone"].nunique())
print("Categorías en destination_zone:", df["destination_zone"].nunique())
print("Categorías en corridor (origen→destino):", df["corridor"].nunique())

# TODO: decisión de encoding (one-hot si son pocas categorías, target/frequency
# encoding si son muchas, o agrupar las de baja frecuencia en "otras") + justificación.
#
# ADVERTENCIA DE LEAKAGE si eligen target encoding: el encoding debe ajustarse
# SOLO con los datos de entrenamiento, después del split train/test — nunca
# sobre el dataset completo aquí en el EDA. historical_corridor_mean_duration
# ya respeta el orden temporal (sección 1.5), pero un target encoding adicional
# sobre 'corridor' tendría que respetarlo también.

In [ ]:
# APLICAR encoding (default razonable: frequency encoding — no usa la
# variable objetivo, así que no tiene riesgo de leakage, a diferencia de
# target encoding. Si deciden usar target encoding porque da mejor señal,
# debe ajustarse SOLO con el split de entrenamiento, no aquí en el EDA).
for col in ["origin_zone", "destination_zone", "corridor"]:
    freq = df_model[col].value_counts(normalize=True)
    df_model[f"{col}_freq"] = df_model[col].map(freq)

df_model[["origin_zone", "origin_zone_freq", "destination_zone", "destination_zone_freq"]].head()

### 4.4 Selección de características

*(Conecta con el Objetivo 2.1 del avance: elegir las características más relevantes para reducir la dimensionalidad y aumentar la capacidad de generalización del modelo.)*

**Permitidas (disponibles al momento de iniciar el viaje):** `distance_km`, `hour_of_day`, `day_of_week`, `origin_zone`, `destination_zone`, `previous_trip_duration`, `historical_corridor_mean_duration`.

**Excluidas explícitamente — resultado del viaje, no se conocen al iniciar:** `average_speed`, `max_speed`, `stops_count`, `stopped_seconds`, `ended_at`, `duration_seconds` (target, no feature).

**`data_source` también queda excluida como feature del modelo** — es metadato de nuestro propio pipeline (distingue real de sintético para fines de este proyecto), no una variable que exista en la operación real: cuando el modelo esté en producción, todo dato que reciba será "real", así que esta columna no le aporta información generalizable.

In [ ]:
# Mutual information — captura relaciones no lineales que Pearson/Spearman
# podrían no reflejar bien. Codificamos categóricas solo para este cálculo
# (no es el encoding final del modelo, solo diagnóstico).
features_candidatas = ["distance_km", "hour_of_day", "day_of_week",
                        "previous_trip_duration", "historical_corridor_mean_duration"]
features_candidatas = [f for f in features_candidatas if f in df.columns]

df_mi = df[features_candidatas + ["origin_zone", "destination_zone", "duration_seconds"]].dropna()
df_mi["origin_zone_enc"] = pd.factorize(df_mi["origin_zone"])[0]
df_mi["destination_zone_enc"] = pd.factorize(df_mi["destination_zone"])[0]

X_mi = df_mi[features_candidatas + ["origin_zone_enc", "destination_zone_enc"]]
y_mi = df_mi["duration_seconds"]

# discrete_features: le decimos a sklearn cuáles columnas son categóricas/
# discretas (hour_of_day, day_of_week, las zonas factorizadas) en vez de
# tratarlas como continuas — mejora la estimación de MI para esas variables.
discrete_mask = [c in ("hour_of_day", "day_of_week", "origin_zone_enc", "destination_zone_enc")
                  for c in X_mi.columns]

mi_scores = mutual_info_regression(X_mi, y_mi, discrete_features=discrete_mask, random_state=42)
mi_result = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)
display(mi_result)

# TODO: ¿alguna feature candidata tiene mutual information cercana a cero?
# Eso es evidencia de que aporta poco y podría descartarse.

In [ ]:
# Colinealidad entre las features candidatas numéricas — si dos están muy
# correlacionadas entre sí (no con el target), una de las dos es redundante.
corr_features = df[[c for c in features_candidatas if df[c].dtype != "object"]].corr()
display(corr_features)

# TODO: ¿alguna pareja tiene correlación > 0.8 entre sí? Por ejemplo,
# distance_km e historical_corridor_mean_duration podrían estar muy
# relacionadas si los corredores largos siempre tardan más — en ese caso,
# ¿cuál de las dos conviene conservar, y por qué?

### 4.5 Guardar dataset limpio

El Avance 2 debe partir de aquí, no repetir estos pasos.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df_model.to_csv("../data/processed/trips_clean.csv", index=False)
print(f"Guardado: ../data/processed/trips_clean.csv ({len(df_model)} filas, ya con faltantes/atípicos/encoding aplicados)")

## 5. Conclusiones

*(Criterio de rúbrica — 20 pts: ideas/observaciones clave, respondiendo todas las preguntas propuestas que apliquen al proyecto.)*

Responder en prosa, no solo como lista.

- **¿Hay valores faltantes? ¿patrones de ausencia?** TODO
- **¿Estadísticas resumidas relevantes?** TODO
- **¿Valores atípicos?** TODO
- **¿Cardinalidad de las variables categóricas?** TODO
- **¿Distribuciones sesgadas? ¿transformación necesaria?** TODO
- **¿Tendencias temporales?** (sí aplica — el generador sintético simula un factor de tráfico por hora/día) TODO
- **¿Correlación entre variables dependientes e independientes?** TODO
- **¿Distribución por categorías (bivariado)?** TODO
- **¿Normalizar imágenes?** No aplica — este proyecto no usa datos de imagen.
- **¿Desequilibrio de clases en la variable objetivo?** No aplica directamente — `duration_seconds` es una variable continua (problema de regresión, no de clasificación).

## Bibliografía

Visengeriyeva, L., Kammer, A., Bär, I., Kniesz, A., y Plöd, M. (2023). *CRISP-ML(Q). The ML Lifecycle Process*. MLOps. INNOQ. https://ml-ops.org/content/crisp-ml

Kumar Mukhiya, S., y Ahmed, U. (2020). *Hands-On Exploratory Data Analysis with Python*. Packt Publishing.

[COMPLETAR: agregar fuentes adicionales citadas, en formato APA 7]